In [1]:
import io
import math
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple, Union, Dict

import fitz
from PIL import Image, ImageTk, ImageGrab, ImageOps, ImageDraw, ImageFont, ImageChops

try:
    from tkinterdnd2 import DND_FILES, TkinterDnD
    DND_AVAILABLE = True
except Exception:
    DND_AVAILABLE = False

import tkinter as tk
from tkinter import ttk, filedialog, messagebox, simpledialog, colorchooser


APP_TITLE = "Figboard"
DEFAULT_CANVAS_W = 1600
DEFAULT_CANVAS_H = 1000
DEFAULT_BG = "white"
DEFAULT_OUTER_MARGIN = 30
DEFAULT_GAP_X = 25
DEFAULT_GAP_Y = 25
DEFAULT_LABEL_FONT_SIZE = 28
DEFAULT_TEXT_FONT_SIZE = 22
HANDLE_SIZE = 10
MIN_PANEL_SIZE = 40
SAFE_GAP = 8
DEFAULT_DPI = 600


def parse_dnd_files(data: str) -> List[str]:
    files = []
    current = ""
    brace = False
    for ch in data:
        if ch == "{":
            brace = True
            current = ""
        elif ch == "}":
            brace = False
            if current:
                files.append(current)
                current = ""
        elif ch == " " and not brace:
            if current:
                files.append(current)
                current = ""
        else:
            current += ch
    if current:
        files.append(current)
    return files


def fit_size_keep_aspect(src_w: int, src_h: int, max_w: int, max_h: int) -> Tuple[int, int]:
    if src_w <= 0 or src_h <= 0:
        return max(1, max_w), max(1, max_h)
    scale = min(max_w / src_w, max_h / src_h)
    return max(1, int(round(src_w * scale))), max(1, int(round(src_h * scale)))


def next_panel_label(index: int) -> str:
    alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    result = ""
    i = index
    while True:
        result = alphabet[i % 26] + result
        i = i // 26 - 1
        if i < 0:
            break
    return result


def list_font_families() -> List[str]:
    root = tk._default_root
    if root is None:
        return ["Arial", "Helvetica", "Times New Roman", "DejaVu Sans"]
    families = sorted(set(root.tk.call("font", "families")))
    preferred = ["Arial", "Helvetica", "Times New Roman", "Calibri", "DejaVu Sans", "Courier New"]
    ordered = [f for f in preferred if f in families] + [f for f in families if f not in preferred]
    return ordered or preferred


def get_font_path_candidates(font_family: str) -> List[str]:
    base = [
        font_family,
        f"{font_family}.ttf",
        f"{font_family}.otf",
    ]
    mapping = {
        "Arial": ["arial.ttf", "Arial.ttf", "LiberationSans-Regular.ttf", "DejaVuSans.ttf"],
        "Helvetica": ["Helvetica.ttf", "Arial.ttf", "LiberationSans-Regular.ttf", "DejaVuSans.ttf"],
        "Times New Roman": ["times.ttf", "Times New Roman.ttf", "LiberationSerif-Regular.ttf", "DejaVuSerif.ttf"],
        "Calibri": ["calibri.ttf", "Carlito-Regular.ttf", "Arial.ttf", "DejaVuSans.ttf"],
        "Courier New": ["cour.ttf", "Courier New.ttf", "LiberationMono-Regular.ttf", "DejaVuSansMono.ttf"],
        "DejaVu Sans": ["DejaVuSans.ttf", "arial.ttf"],
    }
    return mapping.get(font_family, []) + base


def get_font(font_family: str, size: int, bold: bool = False):
    candidates = []
    if bold:
        bold_map = {
            "Arial": ["arialbd.ttf", "Arial Bold.ttf", "LiberationSans-Bold.ttf", "DejaVuSans-Bold.ttf"],
            "Helvetica": ["Helvetica-Bold.ttf", "Arial Bold.ttf", "LiberationSans-Bold.ttf", "DejaVuSans-Bold.ttf"],
            "Times New Roman": ["timesbd.ttf", "Times New Roman Bold.ttf", "LiberationSerif-Bold.ttf", "DejaVuSerif-Bold.ttf"],
            "Calibri": ["calibrib.ttf", "Carlito-Bold.ttf", "Arial Bold.ttf", "DejaVuSans-Bold.ttf"],
            "Courier New": ["courbd.ttf", "Courier New Bold.ttf", "LiberationMono-Bold.ttf", "DejaVuSansMono-Bold.ttf"],
            "DejaVu Sans": ["DejaVuSans-Bold.ttf", "arialbd.ttf"],
        }
        candidates.extend(bold_map.get(font_family, []))
    candidates.extend(get_font_path_candidates(font_family))
    for cand in candidates:
        try:
            return ImageFont.truetype(cand, size=size)
        except Exception:
            continue
    try:
        return ImageFont.truetype("DejaVuSans-Bold.ttf" if bold else "DejaVuSans.ttf", size=size)
    except Exception:
        return ImageFont.load_default()


def clamp(n, lo, hi):
    return max(lo, min(hi, n))


def rects_overlap(a, b, pad=0):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    return not (ax2 + pad <= bx1 or bx2 + pad <= ax1 or ay2 + pad <= by1 or by2 + pad <= ay1)


@dataclass
class PanelItem:
    id: int
    kind: str = "panel"
    source_path: Optional[str] = None
    pil_image: Optional[Image.Image] = None
    original_size: Tuple[int, int] = (100, 100)
    x: int = 50
    y: int = 50
    w: int = 200
    h: int = 150
    label: str = ""
    show_label: bool = True
    label_font_size: int = DEFAULT_LABEL_FONT_SIZE
    label_font_family: str = "Arial"
    label_offset_x: int = 10
    label_offset_y: int = 10
    border_width: int = 0
    border_color: str = "black"
    z_index: int = 0
    tk_preview: Optional[ImageTk.PhotoImage] = None
    group_id: Optional[int] = None

    def bbox(self):
        return self.x, self.y, self.x + self.w, self.y + self.h

    def contains(self, px: int, py: int) -> bool:
        return self.x <= px <= self.x + self.w and self.y <= py <= self.y + self.h

    def resize_handle_hit(self, px: int, py: int) -> bool:
        return (self.x + self.w - HANDLE_SIZE <= px <= self.x + self.w + HANDLE_SIZE and
                self.y + self.h - HANDLE_SIZE <= py <= self.y + self.h + HANDLE_SIZE)


@dataclass
class TextItem:
    id: int
    kind: str = "text"
    text: str = "Text"
    x: int = 60
    y: int = 60
    w: int = 250
    h: int = 80
    font_size: int = DEFAULT_TEXT_FONT_SIZE
    font_family: str = "Arial"
    bold: bool = False
    fill: str = "black"
    align: str = "left"
    background: Optional[str] = None
    outline: Optional[str] = None
    z_index: int = 0
    group_id: Optional[int] = None

    def bbox(self):
        return self.x, self.y, self.x + self.w, self.y + self.h

    def contains(self, px: int, py: int) -> bool:
        return self.x <= px <= self.x + self.w and self.y <= py <= self.y + self.h

    def resize_handle_hit(self, px: int, py: int) -> bool:
        return (self.x + self.w - HANDLE_SIZE <= px <= self.x + self.w + HANDLE_SIZE and
                self.y + self.h - HANDLE_SIZE <= py <= self.y + self.h + HANDLE_SIZE)


CanvasItem = Union[PanelItem, TextItem]


class ScrollableFrame(ttk.Frame):
    def __init__(self, parent, width=260):
        super().__init__(parent)
        self.canvas = tk.Canvas(self, highlightthickness=0, width=width)
        self.vsb = ttk.Scrollbar(self, orient="vertical", command=self.canvas.yview)
        self.inner = ttk.Frame(self.canvas)

        self.inner.bind("<Configure>", self._on_frame_configure)
        self.window_id = self.canvas.create_window((0, 0), window=self.inner, anchor="nw")
        self.canvas.configure(yscrollcommand=self.vsb.set)

        self.canvas.pack(side="left", fill="both", expand=True)
        self.vsb.pack(side="right", fill="y")

        self.canvas.bind("<Configure>", self._on_canvas_configure)
        self.canvas.bind_all("<MouseWheel>", self._on_mousewheel, add="+")
        self.canvas.bind_all("<Button-4>", self._on_mousewheel_linux, add="+")
        self.canvas.bind_all("<Button-5>", self._on_mousewheel_linux, add="+")
        self.inner.bind("<Enter>", lambda e: self.canvas.focus_set())

    def _on_frame_configure(self, _event=None):
        self.canvas.configure(scrollregion=self.canvas.bbox("all"))

    def _on_canvas_configure(self, event):
        self.canvas.itemconfigure(self.window_id, width=event.width)

    def _on_mousewheel(self, event):
        if self.winfo_ismapped():
            try:
                widget = self.winfo_containing(event.x_root, event.y_root)
            except Exception:
                widget = None
            if widget and str(widget).startswith(str(self)):
                delta = -1 * int(event.delta / 120) if event.delta else 0
                if delta:
                    self.canvas.yview_scroll(delta, "units")

    def _on_mousewheel_linux(self, event):
        if self.winfo_ismapped():
            try:
                widget = self.winfo_containing(event.x_root, event.y_root)
            except Exception:
                widget = None
            if widget and str(widget).startswith(str(self)):
                if event.num == 4:
                    self.canvas.yview_scroll(-1, "units")
                elif event.num == 5:
                    self.canvas.yview_scroll(1, "units")


class FigureBoardApp:
    def __init__(self, root: tk.Tk):
        self.root = root
        self.root.title(APP_TITLE)
        self.root.geometry("1800x1080")

        self.canvas_w = DEFAULT_CANVAS_W
        self.canvas_h = DEFAULT_CANVAS_H
        self.bg_color = DEFAULT_BG

        self.items: List[CanvasItem] = []
        self.next_id = 1
        self.next_group_id = 1

        self.selected_ids: List[int] = []
        self.anchor_selected_id: Optional[int] = None

        self.drag_mode: Optional[str] = None
        self.drag_start = (0, 0)
        self.drag_origin_map: Dict[int, Tuple[int, int, int, int]] = {}

        self.erase_mode = False
        self.erase_start = None
        self.erase_current = None

        self.undo_stack = []
        self.redo_stack = []

        self.outer_margin = tk.IntVar(value=DEFAULT_OUTER_MARGIN)
        self.gap_x = tk.IntVar(value=DEFAULT_GAP_X)
        self.gap_y = tk.IntVar(value=DEFAULT_GAP_Y)
        self.default_label_size = tk.IntVar(value=DEFAULT_LABEL_FONT_SIZE)
        self.default_text_size = tk.IntVar(value=DEFAULT_TEXT_FONT_SIZE)
        self.default_font_family = tk.StringVar(value="Arial")
        self.auto_trim_on_export = tk.BooleanVar(value=True)
        self.keep_label_order = tk.BooleanVar(value=True)
        self.show_grid = tk.BooleanVar(value=False)
        self.canvas_unit = tk.StringVar(value="px")
        self.export_dpi = tk.IntVar(value=DEFAULT_DPI)
        self.preview_scale = 1.0

        self.status_var = tk.StringVar(value="Ready")

        self.font_families = list_font_families()

        self._build_ui()
        self._bind_events()
        self.redraw()

    def _build_ui(self):
        outer = ttk.Frame(self.root)
        outer.pack(fill="both", expand=True)

        left_wrap = ttk.Frame(outer, width=300)
        left_wrap.pack(side="left", fill="y", padx=6, pady=6)

        center = ttk.Frame(outer)
        center.pack(side="left", fill="both", expand=True, padx=6, pady=6)

        right_wrap = ttk.Frame(outer, width=330)
        right_wrap.pack(side="right", fill="y", padx=6, pady=6)

        self.left_scroll = ScrollableFrame(left_wrap, width=300)
        self.left_scroll.pack(fill="both", expand=True)
        left = self.left_scroll.inner

        self.right_scroll = ScrollableFrame(right_wrap, width=330)
        self.right_scroll.pack(fill="both", expand=True)
        right = self.right_scroll.inner

        import_box = ttk.LabelFrame(left, text="Import")
        import_box.pack(fill="x", pady=(0, 8))
        ttk.Button(import_box, text="Add image/PDF files", command=self.add_files).pack(fill="x", padx=6, pady=4)
        ttk.Button(import_box, text="Paste from clipboard", command=self.paste_from_clipboard).pack(fill="x", padx=6, pady=4)
        ttk.Button(import_box, text="Add text box", command=self.add_text_box).pack(fill="x", padx=6, pady=4)
        ttk.Button(import_box, text="Add caption box", command=lambda: self.add_text_box(default_text="Caption")).pack(fill="x", padx=6, pady=4)
        ttk.Button(import_box, text="Add auto caption below panels", command=self.add_caption_below_last_figure).pack(fill="x", padx=6, pady=4)

        arrange_box = ttk.LabelFrame(left, text="Arrange")
        arrange_box.pack(fill="x", pady=(0, 8))
        grid_row = ttk.Frame(arrange_box)
        grid_row.pack(fill="x", padx=6, pady=(4, 2))
        ttk.Label(grid_row, text="Rows").pack(side="left")
        self.grid_rows_entry = ttk.Entry(grid_row, width=6)
        self.grid_rows_entry.insert(0, "2")
        self.grid_rows_entry.pack(side="left", padx=(4, 12))
        ttk.Label(grid_row, text="Cols").pack(side="left")
        self.grid_cols_entry = ttk.Entry(grid_row, width=6)
        self.grid_cols_entry.insert(0, "2")
        self.grid_cols_entry.pack(side="left", padx=4)
        ttk.Button(arrange_box, text="Apply custom grid", command=self.apply_custom_grid).pack(fill="x", padx=6, pady=4)
        ttk.Button(arrange_box, text="Auto grid", command=self.auto_grid).pack(fill="x", padx=6, pady=4)
        ttk.Button(arrange_box, text="Distribute horizontally", command=self.distribute_h).pack(fill="x", padx=6, pady=4)
        ttk.Button(arrange_box, text="Distribute vertically", command=self.distribute_v).pack(fill="x", padx=6, pady=4)
        ttk.Button(arrange_box, text="Align left", command=lambda: self.align_to_anchor("left")).pack(fill="x", padx=6, pady=4)
        ttk.Button(arrange_box, text="Align right", command=lambda: self.align_to_anchor("right")).pack(fill="x", padx=6, pady=4)
        ttk.Button(arrange_box, text="Align top", command=lambda: self.align_to_anchor("top")).pack(fill="x", padx=6, pady=4)
        ttk.Button(arrange_box, text="Align bottom", command=lambda: self.align_to_anchor("bottom")).pack(fill="x", padx=6, pady=4)
        ttk.Button(arrange_box, text="Same widths", command=self.same_widths).pack(fill="x", padx=6, pady=4)
        ttk.Button(arrange_box, text="Same heights", command=self.same_heights).pack(fill="x", padx=6, pady=4)
        ttk.Button(arrange_box, text="Tight crop to content", command=self.crop_canvas_to_content).pack(fill="x", padx=6, pady=4)
        ttk.Button(arrange_box, text="Resolve overlaps", command=self.resolve_all_overlaps).pack(fill="x", padx=6, pady=4)

        group_box = ttk.LabelFrame(left, text="Selection and groups")
        group_box.pack(fill="x", pady=(0, 8))
        ttk.Button(group_box, text="Select all", command=self.select_all_items).pack(fill="x", padx=6, pady=4)
        ttk.Button(group_box, text="Group selected", command=self.group_selected).pack(fill="x", padx=6, pady=4)
        ttk.Button(group_box, text="Ungroup selected", command=self.ungroup_selected).pack(fill="x", padx=6, pady=4)
        ttk.Button(group_box, text="Duplicate selected", command=self.duplicate_selected).pack(fill="x", padx=6, pady=4)
        ttk.Button(group_box, text="Delete selected", command=self.delete_selected).pack(fill="x", padx=6, pady=4)

        edit_box = ttk.LabelFrame(left, text="Edit")
        edit_box.pack(fill="x", pady=(0, 8))
        ttk.Button(edit_box, text="Undo", command=self.undo).pack(fill="x", padx=6, pady=4)
        ttk.Button(edit_box, text="Redo", command=self.redo).pack(fill="x", padx=6, pady=4)
        ttk.Button(edit_box, text="Toggle erase rectangle mode", command=self.toggle_erase_mode).pack(fill="x", padx=6, pady=4)

        label_box = ttk.LabelFrame(left, text="Labels")
        label_box.pack(fill="x", pady=(0, 8))
        ttk.Button(label_box, text="Regenerate A, B, C...", command=self.regenerate_labels).pack(fill="x", padx=6, pady=4)
        ttk.Button(label_box, text="Toggle labels on/off", command=self.toggle_labels).pack(fill="x", padx=6, pady=4)
        ttk.Label(label_box, text="Default label size").pack(anchor="w", padx=6, pady=(6, 2))
        ttk.Spinbox(label_box, from_=8, to=120, textvariable=self.default_label_size, width=8,
                    command=self.apply_default_label_size).pack(anchor="w", padx=6, pady=(0, 6))
        ttk.Label(label_box, text="Default font family").pack(anchor="w", padx=6, pady=(2, 2))
        self.default_font_combo = ttk.Combobox(label_box, values=self.font_families, textvariable=self.default_font_family, state="readonly")
        self.default_font_combo.pack(fill="x", padx=6, pady=(0, 6))
        ttk.Checkbutton(label_box, text="Keep labels by add order", variable=self.keep_label_order).pack(anchor="w", padx=6, pady=4)

        canvas_box = ttk.LabelFrame(left, text="Canvas")
        canvas_box.pack(fill="x", pady=(0, 8))
        row = ttk.Frame(canvas_box)
        row.pack(fill="x", padx=6, pady=4)
        ttk.Label(row, text="Unit").pack(side="left")
        ttk.Combobox(row, values=["px", "in", "cm"], textvariable=self.canvas_unit, width=6, state="readonly").pack(side="left", padx=(4, 10))
        ttk.Label(row, text="W").pack(side="left")
        self.canvas_w_entry = ttk.Entry(row, width=8)
        self.canvas_w_entry.insert(0, str(self.canvas_w))
        self.canvas_w_entry.pack(side="left", padx=(4, 10))
        ttk.Label(row, text="H").pack(side="left")
        self.canvas_h_entry = ttk.Entry(row, width=8)
        self.canvas_h_entry.insert(0, str(self.canvas_h))
        self.canvas_h_entry.pack(side="left", padx=4)
        ttk.Button(canvas_box, text="Apply canvas size", command=self.apply_canvas_size).pack(fill="x", padx=6, pady=4)

        ttk.Label(canvas_box, text="Outer margin").pack(anchor="w", padx=6, pady=(6, 2))
        ttk.Spinbox(canvas_box, from_=0, to=1000, textvariable=self.outer_margin, width=8, command=self.redraw).pack(anchor="w", padx=6, pady=(0, 6))
        ttk.Label(canvas_box, text="Gap X").pack(anchor="w", padx=6, pady=(2, 2))
        ttk.Spinbox(canvas_box, from_=0, to=1000, textvariable=self.gap_x, width=8).pack(anchor="w", padx=6, pady=(0, 6))
        ttk.Label(canvas_box, text="Gap Y").pack(anchor="w", padx=6, pady=(2, 2))
        ttk.Spinbox(canvas_box, from_=0, to=1000, textvariable=self.gap_y, width=8).pack(anchor="w", padx=6, pady=(0, 6))
        ttk.Checkbutton(canvas_box, text="Show grid", variable=self.show_grid, command=self.redraw).pack(anchor="w", padx=6, pady=4)

        export_box = ttk.LabelFrame(left, text="Export")
        export_box.pack(fill="x", pady=(0, 8))
        ttk.Checkbutton(export_box, text="Trim whitespace on export", variable=self.auto_trim_on_export).pack(anchor="w", padx=6, pady=4)
        dpi_row = ttk.Frame(export_box)
        dpi_row.pack(fill="x", padx=6, pady=4)
        ttk.Label(dpi_row, text="DPI").pack(side="left")
        ttk.Spinbox(dpi_row, from_=72, to=2400, textvariable=self.export_dpi, width=8).pack(side="left", padx=6)
        ttk.Button(export_box, text="Export PNG", command=lambda: self.export_image("PNG")).pack(fill="x", padx=6, pady=4)
        ttk.Button(export_box, text="Export TIFF", command=lambda: self.export_image("TIFF")).pack(fill="x", padx=6, pady=4)
        ttk.Button(export_box, text="Export PDF", command=self.export_pdf).pack(fill="x", padx=6, pady=4)

        canvas_frame = ttk.Frame(center)
        canvas_frame.pack(fill="both", expand=True)

        self.hbar = ttk.Scrollbar(canvas_frame, orient="horizontal")
        self.vbar = ttk.Scrollbar(canvas_frame, orient="vertical")
        self.canvas = tk.Canvas(canvas_frame, bg=self.bg_color,
                                xscrollcommand=self.hbar.set,
                                yscrollcommand=self.vbar.set,
                                highlightthickness=1,
                                highlightbackground="#888")
        self.hbar.config(command=self.canvas.xview)
        self.vbar.config(command=self.canvas.yview)

        self.hbar.pack(side="bottom", fill="x")
        self.vbar.pack(side="right", fill="y")
        self.canvas.pack(side="left", fill="both", expand=True)
        self.canvas.config(scrollregion=(0, 0, self.canvas_w, self.canvas_h))

        if DND_AVAILABLE:
            try:
                self.canvas.drop_target_register(DND_FILES)
                self.canvas.dnd_bind("<<Drop>>", self.on_drop)
            except Exception:
                pass

        prop_box = ttk.LabelFrame(right, text="Selected item")
        prop_box.pack(fill="x", pady=(0, 8))

        self.sel_type_var = tk.StringVar(value="-")
        self.sel_label_var = tk.StringVar(value="")
        self.sel_text_var = tk.StringVar(value="")
        self.sel_x_var = tk.StringVar(value="0")
        self.sel_y_var = tk.StringVar(value="0")
        self.sel_w_var = tk.StringVar(value="0")
        self.sel_h_var = tk.StringVar(value="0")
        self.sel_font_var = tk.StringVar(value=str(DEFAULT_TEXT_FONT_SIZE))
        self.sel_font_family_var = tk.StringVar(value="Arial")
        self.sel_show_label_var = tk.BooleanVar(value=True)
        self.sel_align_var = tk.StringVar(value="left")

        ttk.Label(prop_box, text="Type").grid(row=0, column=0, sticky="w", padx=6, pady=4)
        ttk.Label(prop_box, textvariable=self.sel_type_var).grid(row=0, column=1, sticky="w", padx=6, pady=4)

        ttk.Label(prop_box, text="Label").grid(row=1, column=0, sticky="w", padx=6, pady=4)
        ttk.Entry(prop_box, textvariable=self.sel_label_var).grid(row=1, column=1, sticky="ew", padx=6, pady=4)

        ttk.Label(prop_box, text="Text").grid(row=2, column=0, sticky="w", padx=6, pady=4)
        ttk.Entry(prop_box, textvariable=self.sel_text_var).grid(row=2, column=1, sticky="ew", padx=6, pady=4)

        ttk.Label(prop_box, text="X").grid(row=3, column=0, sticky="w", padx=6, pady=4)
        ttk.Entry(prop_box, textvariable=self.sel_x_var, width=10).grid(row=3, column=1, sticky="w", padx=6, pady=4)

        ttk.Label(prop_box, text="Y").grid(row=4, column=0, sticky="w", padx=6, pady=4)
        ttk.Entry(prop_box, textvariable=self.sel_y_var, width=10).grid(row=4, column=1, sticky="w", padx=6, pady=4)

        ttk.Label(prop_box, text="W").grid(row=5, column=0, sticky="w", padx=6, pady=4)
        ttk.Entry(prop_box, textvariable=self.sel_w_var, width=10).grid(row=5, column=1, sticky="w", padx=6, pady=4)

        ttk.Label(prop_box, text="H").grid(row=6, column=0, sticky="w", padx=6, pady=4)
        ttk.Entry(prop_box, textvariable=self.sel_h_var, width=10).grid(row=6, column=1, sticky="w", padx=6, pady=4)

        ttk.Label(prop_box, text="Font size").grid(row=7, column=0, sticky="w", padx=6, pady=4)
        ttk.Entry(prop_box, textvariable=self.sel_font_var, width=10).grid(row=7, column=1, sticky="w", padx=6, pady=4)

        ttk.Label(prop_box, text="Font family").grid(row=8, column=0, sticky="w", padx=6, pady=4)
        ttk.Combobox(prop_box, values=self.font_families, textvariable=self.sel_font_family_var, state="readonly").grid(row=8, column=1, sticky="ew", padx=6, pady=4)

        ttk.Label(prop_box, text="Align").grid(row=9, column=0, sticky="w", padx=6, pady=4)
        ttk.Combobox(prop_box, values=["left", "center", "right"], textvariable=self.sel_align_var, state="readonly").grid(row=9, column=1, sticky="ew", padx=6, pady=4)

        ttk.Checkbutton(prop_box, text="Show panel label", variable=self.sel_show_label_var).grid(
            row=10, column=0, columnspan=2, sticky="w", padx=6, pady=4
        )

        ttk.Button(prop_box, text="Apply changes", command=self.apply_selected_properties).grid(
            row=11, column=0, columnspan=2, sticky="ew", padx=6, pady=6
        )
        ttk.Button(prop_box, text="Bring to front", command=self.bring_to_front).grid(
            row=12, column=0, columnspan=2, sticky="ew", padx=6, pady=4
        )
        ttk.Button(prop_box, text="Send to back", command=self.send_to_back).grid(
            row=13, column=0, columnspan=2, sticky="ew", padx=6, pady=4
        )
        ttk.Button(prop_box, text="Pick text colour", command=self.pick_selected_text_colour).grid(
            row=14, column=0, columnspan=2, sticky="ew", padx=6, pady=4
        )

        prop_box.columnconfigure(1, weight=1)

        help_box = ttk.LabelFrame(right, text="Shortcuts")
        help_box.pack(fill="x", pady=(0, 8))
        help_text = (
            "Ctrl+A  select all\n"
            "Ctrl+Z  undo\n"
            "Ctrl+Y  redo\n"
            "Ctrl+Click  multi-select\n"
            "Delete  remove selected\n"
            "Ctrl+V  paste image\n"
            "Ctrl+O  open files\n"
            "Ctrl+E  export PNG\n"
            "Ctrl+G  auto grid\n"
            "Mouse wheel over canvas  scroll\n"
            "Erase mode: drag rectangle on a panel\n"
        )
        ttk.Label(help_box, text=help_text, justify="left").pack(anchor="w", padx=8, pady=8)

        status = ttk.Label(self.root, textvariable=self.status_var, anchor="w")
        status.pack(fill="x", side="bottom")

    def _bind_events(self):
        self.canvas.bind("<Button-1>", self.on_canvas_press)
        self.canvas.bind("<B1-Motion>", self.on_canvas_drag)
        self.canvas.bind("<ButtonRelease-1>", self.on_canvas_release)
        self.canvas.bind("<Double-Button-1>", self.on_double_click)
        self.canvas.bind("<MouseWheel>", self.on_canvas_mousewheel, add="+")
        self.canvas.bind("<Shift-MouseWheel>", self.on_canvas_mousewheel_horizontal, add="+")
        self.canvas.bind("<Button-4>", self.on_canvas_mousewheel_linux, add="+")
        self.canvas.bind("<Button-5>", self.on_canvas_mousewheel_linux, add="+")
        self.canvas.bind("<Shift-Button-4>", self.on_canvas_mousewheel_linux_horizontal, add="+")
        self.canvas.bind("<Shift-Button-5>", self.on_canvas_mousewheel_linux_horizontal, add="+")
        self.root.bind("<Delete>", lambda e: self.delete_selected())
        self.root.bind("<Control-v>", lambda e: self.paste_from_clipboard())
        self.root.bind("<Control-o>", lambda e: self.add_files())
        self.root.bind("<Control-e>", lambda e: self.export_image("PNG"))
        self.root.bind("<Control-g>", lambda e: self.auto_grid())
        self.root.bind("<Control-a>", lambda e: self.select_all_items())
        self.root.bind("<Control-z>", lambda e: self.undo())
        self.root.bind("<Control-y>", lambda e: self.redo())

    def on_canvas_mousewheel(self, event):
        self.canvas.yview_scroll(-1 * int(event.delta / 120), "units")

    def on_canvas_mousewheel_horizontal(self, event):
        self.canvas.xview_scroll(-1 * int(event.delta / 120), "units")

    def on_canvas_mousewheel_linux(self, event):
        if event.num == 4:
            self.canvas.yview_scroll(-1, "units")
        elif event.num == 5:
            self.canvas.yview_scroll(1, "units")

    def on_canvas_mousewheel_linux_horizontal(self, event):
        if event.num == 4:
            self.canvas.xview_scroll(-1, "units")
        elif event.num == 5:
            self.canvas.xview_scroll(1, "units")

    def set_status(self, text: str):
        self.status_var.set(text)

    def save_undo_state(self):
        state = {
            "canvas_w": self.canvas_w,
            "canvas_h": self.canvas_h,
            "items": self._clone_items(self.items),
            "selected_ids": list(self.selected_ids),
            "anchor_selected_id": self.anchor_selected_id,
            "next_id": self.next_id,
            "next_group_id": self.next_group_id,
        }
        self.undo_stack.append(state)
        if len(self.undo_stack) > 100:
            self.undo_stack.pop(0)
        self.redo_stack.clear()

    def _clone_items(self, items):
        cloned = []
        for item in items:
            if isinstance(item, PanelItem):
                cloned.append(PanelItem(
                    id=item.id,
                    source_path=item.source_path,
                    pil_image=item.pil_image.copy() if item.pil_image else None,
                    original_size=item.original_size,
                    x=item.x, y=item.y, w=item.w, h=item.h,
                    label=item.label, show_label=item.show_label,
                    label_font_size=item.label_font_size,
                    label_font_family=item.label_font_family,
                    label_offset_x=item.label_offset_x,
                    label_offset_y=item.label_offset_y,
                    border_width=item.border_width,
                    border_color=item.border_color,
                    z_index=item.z_index,
                    group_id=item.group_id
                ))
            else:
                cloned.append(TextItem(
                    id=item.id, text=item.text,
                    x=item.x, y=item.y, w=item.w, h=item.h,
                    font_size=item.font_size,
                    font_family=item.font_family,
                    bold=item.bold, fill=item.fill,
                    align=item.align, background=item.background,
                    outline=item.outline, z_index=item.z_index,
                    group_id=item.group_id
                ))
        return cloned

    def restore_state(self, state):
        self.canvas_w = state["canvas_w"]
        self.canvas_h = state["canvas_h"]
        self.items = self._clone_items(state["items"])
        self.selected_ids = list(state["selected_ids"])
        self.anchor_selected_id = state["anchor_selected_id"]
        self.next_id = state["next_id"]
        self.next_group_id = state["next_group_id"]
        self.canvas_w_entry.delete(0, "end")
        self.canvas_w_entry.insert(0, str(self.canvas_w))
        self.canvas_h_entry.delete(0, "end")
        self.canvas_h_entry.insert(0, str(self.canvas_h))
        self.canvas.config(scrollregion=(0, 0, self.canvas_w, self.canvas_h))
        self.refresh_selected_panel()
        self.redraw()

    def undo(self):
        if not self.undo_stack:
            return
        current = {
            "canvas_w": self.canvas_w,
            "canvas_h": self.canvas_h,
            "items": self._clone_items(self.items),
            "selected_ids": list(self.selected_ids),
            "anchor_selected_id": self.anchor_selected_id,
            "next_id": self.next_id,
            "next_group_id": self.next_group_id,
        }
        self.redo_stack.append(current)
        state = self.undo_stack.pop()
        self.restore_state(state)
        self.set_status("Undo")

    def redo(self):
        if not self.redo_stack:
            return
        current = {
            "canvas_w": self.canvas_w,
            "canvas_h": self.canvas_h,
            "items": self._clone_items(self.items),
            "selected_ids": list(self.selected_ids),
            "anchor_selected_id": self.anchor_selected_id,
            "next_id": self.next_id,
            "next_group_id": self.next_group_id,
        }
        self.undo_stack.append(current)
        state = self.redo_stack.pop()
        self.restore_state(state)
        self.set_status("Redo")

    def px_from_unit(self, value: float) -> int:
        unit = self.canvas_unit.get()
        dpi = self.export_dpi.get()
        if unit == "px":
            return int(round(value))
        if unit == "in":
            return int(round(value * dpi))
        if unit == "cm":
            return int(round((value / 2.54) * dpi))
        return int(round(value))

    def unit_from_px(self, px: int) -> float:
        unit = self.canvas_unit.get()
        dpi = self.export_dpi.get()
        if unit == "px":
            return float(px)
        if unit == "in":
            return px / dpi
        if unit == "cm":
            return (px / dpi) * 2.54
        return float(px)

    def apply_canvas_size(self):
        try:
            w_in = float(self.canvas_w_entry.get())
            h_in = float(self.canvas_h_entry.get())
            new_w = max(100, self.px_from_unit(w_in))
            new_h = max(100, self.px_from_unit(h_in))
        except ValueError:
            messagebox.showerror("Invalid size", "Canvas width and height must be numeric.")
            return
        self.save_undo_state()
        self.canvas_w = new_w
        self.canvas_h = new_h
        self.canvas.config(scrollregion=(0, 0, self.canvas_w, self.canvas_h))
        self.redraw()
        self.set_status(f"Canvas set to {self.canvas_w} × {self.canvas_h} px")

    def get_next_id(self) -> int:
        out = self.next_id
        self.next_id += 1
        return out

    def get_next_group_id(self) -> int:
        out = self.next_group_id
        self.next_group_id += 1
        return out

    def add_files(self):
        filetypes = [
            ("Supported", "*.png *.jpg *.jpeg *.tif *.tiff *.bmp *.gif *.pdf"),
            ("Images", "*.png *.jpg *.jpeg *.tif *.tiff *.bmp *.gif"),
            ("PDF files", "*.pdf"),
            ("All files", "*.*"),
        ]
        paths = filedialog.askopenfilenames(title="Select images or PDFs", filetypes=filetypes)
        if not paths:
            return
        self.save_undo_state()
        self.load_paths(list(paths))
        self.redraw()

    def on_drop(self, event):
        paths = parse_dnd_files(event.data)
        if paths:
            self.save_undo_state()
            self.load_paths(paths)
            self.redraw()

    def load_paths(self, paths: List[str]):
        added = 0
        for path in paths:
            path = str(path).strip()
            if not path:
                continue
            ext = Path(path).suffix.lower()
            try:
                if ext == ".pdf":
                    added += self._load_pdf(path)
                else:
                    self._load_image_path(path)
                    added += 1
            except Exception as e:
                messagebox.showerror("Import error", f"Could not open:\n{path}\n\n{e}")
        if added:
            self.regenerate_labels(silent=True)
            self.set_status(f"Added {added} item(s)")
            self.auto_grid_if_reasonable()

    def _load_image_path(self, path: str):
        img = Image.open(path).convert("RGBA")
        self._add_panel_from_pil(img, source_path=path)

    def _load_pdf(self, path: str) -> int:
        doc = fitz.open(path)
        count = 0
        try:
            for pno in range(len(doc)):
                page = doc[pno]
                pix = page.get_pixmap(matrix=fitz.Matrix(4, 4), alpha=False)
                img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGBA")
                self._add_panel_from_pil(img, source_path=f"{path} [page {pno + 1}]")
                count += 1
        finally:
            doc.close()
        return count

    def paste_from_clipboard(self):
        try:
            data = ImageGrab.grabclipboard()
        except Exception as e:
            messagebox.showerror("Clipboard error", f"Could not access clipboard.\n\n{e}")
            return

        if isinstance(data, Image.Image):
            self.save_undo_state()
            self._add_panel_from_pil(data.convert("RGBA"), source_path="clipboard")
            self.regenerate_labels(silent=True)
            self.redraw()
            self.set_status("Pasted image from clipboard")
            self.auto_grid_if_reasonable()
            return

        if isinstance(data, list):
            self.save_undo_state()
            self.load_paths([str(p) for p in data])
            self.redraw()
            return

        messagebox.showinfo("Clipboard", "No image or file list found in clipboard.")

    def _add_panel_from_pil(self, img: Image.Image, source_path: Optional[str] = None):
        ow, oh = img.size
        w, h = fit_size_keep_aspect(ow, oh, 350, 260)
        offset = 20 * (len(self.items) % 10)
        panel = PanelItem(
            id=self.get_next_id(),
            source_path=source_path,
            pil_image=img,
            original_size=(ow, oh),
            x=50 + offset,
            y=50 + offset,
            w=w,
            h=h,
            label="",
            show_label=True,
            label_font_size=self.default_label_size.get(),
            label_font_family=self.default_font_family.get(),
            z_index=len(self.items),
        )
        self.items.append(panel)
        self.selected_ids = [panel.id]
        self.anchor_selected_id = panel.id

    def add_text_box(self, default_text="Text"):
        self.save_undo_state()
        item = TextItem(
            id=self.get_next_id(),
            text=default_text,
            x=60 + 15 * (len(self.items) % 10),
            y=60 + 15 * (len(self.items) % 10),
            w=320,
            h=90,
            font_size=self.default_text_size.get(),
            font_family=self.default_font_family.get(),
            z_index=len(self.items),
        )
        self.items.append(item)
        self.selected_ids = [item.id]
        self.anchor_selected_id = item.id
        self.refresh_selected_panel()
        self.redraw()
        self.set_status("Added text box")

    def add_caption_below_last_figure(self):
        panels = [i for i in self.items if isinstance(i, PanelItem)]
        if not panels:
            return
        self.save_undo_state()
        x1 = min(p.x for p in panels)
        x2 = max(p.x + p.w for p in panels)
        y2 = max(p.y + p.h for p in panels)
        caption = TextItem(
            id=self.get_next_id(),
            text="Caption",
            x=x1,
            y=y2 + self.gap_y.get(),
            w=max(300, x2 - x1),
            h=120,
            font_size=self.default_text_size.get(),
            font_family=self.default_font_family.get(),
            z_index=len(self.items),
        )
        self.items.append(caption)
        needed_h = caption.y + caption.h + self.outer_margin.get()
        if needed_h > self.canvas_h:
            self.canvas_h = needed_h
            self.canvas_h_entry.delete(0, "end")
            self.canvas_h_entry.insert(0, str(self.canvas_h))
            self.canvas.config(scrollregion=(0, 0, self.canvas_w, self.canvas_h))
        self.selected_ids = [caption.id]
        self.anchor_selected_id = caption.id
        self.refresh_selected_panel()
        self.redraw()
        self.set_status("Added caption below panels")

    def redraw(self):
        self.canvas.delete("all")
        self.canvas.config(scrollregion=(0, 0, self.canvas_w, self.canvas_h))
        self.canvas.create_rectangle(0, 0, self.canvas_w, self.canvas_h, fill=self.bg_color, outline="")
        self._draw_usable_margin_frame()
        if self.show_grid.get():
            self._draw_grid()
        for item in sorted(self.items, key=lambda x: x.z_index):
            if isinstance(item, PanelItem):
                self._draw_panel(item)
            else:
                self._draw_text_item(item)
        for item in self.get_selected_items():
            self._draw_selection(item, is_anchor=(item.id == self.anchor_selected_id))
        if self.erase_mode and self.erase_start and self.erase_current:
            x1, y1 = self.erase_start
            x2, y2 = self.erase_current
            self.canvas.create_rectangle(x1, y1, x2, y2, outline="red", dash=(6, 3), width=2)

    def _draw_usable_margin_frame(self):
        m = self.outer_margin.get()
        self.canvas.create_rectangle(
            m, m, self.canvas_w - m, self.canvas_h - m,
            outline="#a0a0a0", dash=(4, 3), width=1
        )

    def _draw_grid(self):
        step = 50
        for x in range(0, self.canvas_w + 1, step):
            self.canvas.create_line(x, 0, x, self.canvas_h, fill="#eeeeee")
        for y in range(0, self.canvas_h + 1, step):
            self.canvas.create_line(0, y, self.canvas_w, y, fill="#eeeeee")

    def _draw_panel(self, item: PanelItem):
        if item.pil_image is None:
            return
        preview = item.pil_image.resize((max(1, item.w), max(1, item.h)), Image.LANCZOS)
        item.tk_preview = ImageTk.PhotoImage(preview)
        self.canvas.create_image(item.x, item.y, image=item.tk_preview, anchor="nw")
        if item.border_width > 0:
            self.canvas.create_rectangle(item.x, item.y, item.x + item.w, item.y + item.h,
                                         outline=item.border_color, width=item.border_width)
        if item.show_label and item.label:
            font_style = (item.label_font_family, item.label_font_size, "bold")
            self.canvas.create_text(
                item.x + item.label_offset_x,
                item.y + item.label_offset_y,
                text=item.label,
                anchor="nw",
                font=font_style,
                fill="black"
            )

    def _draw_text_item(self, item: TextItem):
        if item.background:
            self.canvas.create_rectangle(item.x, item.y, item.x + item.w, item.y + item.h,
                                         fill=item.background, outline=item.outline or "")
        elif item.outline:
            self.canvas.create_rectangle(item.x, item.y, item.x + item.w, item.y + item.h,
                                         outline=item.outline, width=1)
        anchor_map = {"left": "nw", "center": "n", "right": "ne"}
        tx = item.x + 6 if item.align == "left" else item.x + item.w // 2 if item.align == "center" else item.x + item.w - 6
        self.canvas.create_text(
            tx, item.y + 6,
            text=item.text,
            anchor=anchor_map.get(item.align, "nw"),
            width=max(20, item.w - 12),
            font=(item.font_family, item.font_size, "bold" if item.bold else "normal"),
            fill=item.fill
        )

    def _draw_selection(self, item: CanvasItem, is_anchor=False):
        x1, y1, x2, y2 = item.bbox()
        outline = "#ff7a00" if is_anchor else "#2a73ff"
        self.canvas.create_rectangle(x1, y1, x2, y2, outline=outline, width=2, dash=(5, 3))
        self.canvas.create_rectangle(x2 - HANDLE_SIZE, y2 - HANDLE_SIZE, x2, y2, fill=outline, outline=outline)

    def get_item_at(self, x: int, y: int) -> Optional[CanvasItem]:
        for item in sorted(self.items, key=lambda z: z.z_index, reverse=True):
            if item.contains(x, y):
                return item
        return None

    def get_item_by_id(self, item_id: int) -> Optional[CanvasItem]:
        for item in self.items:
            if item.id == item_id:
                return item
        return None

    def get_selected_items(self) -> List[CanvasItem]:
        result = []
        for item_id in self.selected_ids:
            item = self.get_item_by_id(item_id)
            if item is not None:
                result.append(item)
        return result

    def select_all_items(self):
        self.selected_ids = [item.id for item in self.items]
        self.anchor_selected_id = self.selected_ids[-1] if self.selected_ids else None
        self.refresh_selected_panel()
        self.redraw()
        self.set_status("Selected all items")

    def on_canvas_press(self, event):
        x = int(self.canvas.canvasx(event.x))
        y = int(self.canvas.canvasy(event.y))

        if self.erase_mode:
            item = self.get_item_at(x, y)
            if isinstance(item, PanelItem):
                self.save_undo_state()
                self.erase_start = (x, y)
                self.erase_current = (x, y)
            return

        item = self.get_item_at(x, y)
        ctrl = bool(event.state & 0x0004)

        self.drag_start = (x, y)
        self.drag_mode = None
        self.drag_origin_map = {}

        if item is None:
            if not ctrl:
                self.selected_ids = []
                self.anchor_selected_id = None
                self.refresh_selected_panel()
                self.redraw()
            return

        if ctrl:
            if item.id in self.selected_ids:
                self.selected_ids = [i for i in self.selected_ids if i != item.id]
                if self.anchor_selected_id == item.id:
                    self.anchor_selected_id = self.selected_ids[-1] if self.selected_ids else None
                self.refresh_selected_panel()
                self.redraw()
                return
            else:
                self.selected_ids.append(item.id)
                self.anchor_selected_id = item.id
        else:
            if item.id not in self.selected_ids or len(self.selected_ids) != 1:
                self.selected_ids = [item.id]
            self.anchor_selected_id = item.id

        selected_items = self.get_selected_items()
        hit_resize = item.resize_handle_hit(x, y)
        if hit_resize and len(selected_items) == 1:
            self.drag_mode = "resize"
            self.drag_origin_map[item.id] = (item.x, item.y, item.w, item.h)
        else:
            self.drag_mode = "move"
            for sel in selected_items:
                self.drag_origin_map[sel.id] = (sel.x, sel.y, sel.w, sel.h)

        self.refresh_selected_panel()
        self.redraw()

    def on_canvas_drag(self, event):
        x = int(self.canvas.canvasx(event.x))
        y = int(self.canvas.canvasy(event.y))

        if self.erase_mode:
            if self.erase_start:
                self.erase_current = (x, y)
                self.redraw()
            return

        if not self.drag_mode or not self.drag_origin_map:
            return

        dx = x - self.drag_start[0]
        dy = y - self.drag_start[1]

        if self.drag_mode == "move":
            for item_id, geom in self.drag_origin_map.items():
                item = self.get_item_by_id(item_id)
                if item is None:
                    continue
                ox, oy, ow, oh = geom
                item.x = max(0, ox + dx)
                item.y = max(0, oy + dy)
        elif self.drag_mode == "resize":
            item_id = next(iter(self.drag_origin_map.keys()))
            item = self.get_item_by_id(item_id)
            if item is not None:
                ox, oy, ow, oh = self.drag_origin_map[item_id]
                item.w = max(MIN_PANEL_SIZE, ow + dx)
                item.h = max(MIN_PANEL_SIZE, oh + dy)

        self.refresh_selected_panel(live=True)
        self.redraw()

    def on_canvas_release(self, event):
        x = int(self.canvas.canvasx(event.x))
        y = int(self.canvas.canvasy(event.y))

        if self.erase_mode and self.erase_start:
            x1, y1 = self.erase_start
            x2, y2 = x, y
            self.apply_erase_rectangle(x1, y1, x2, y2)
            self.erase_start = None
            self.erase_current = None
            self.redraw()
            return

        if self.drag_mode:
            self.save_after_live_change()
        self.drag_mode = None
        self.drag_origin_map = {}

    def save_after_live_change(self):
        if not self.undo_stack:
            return

    def on_double_click(self, event):
        x = int(self.canvas.canvasx(event.x))
        y = int(self.canvas.canvasy(event.y))
        item = self.get_item_at(x, y)
        if item is None:
            return
        self.save_undo_state()
        if isinstance(item, TextItem):
            new_text = simpledialog.askstring("Edit text", "Text:", initialvalue=item.text, parent=self.root)
            if new_text is not None:
                item.text = new_text
                if item.text.strip().lower().startswith("caption"):
                    self.fit_text_box_height(item)
                self.refresh_selected_panel()
                self.redraw()
        elif isinstance(item, PanelItem):
            new_label = simpledialog.askstring("Edit label", "Panel label:", initialvalue=item.label, parent=self.root)
            if new_label is not None:
                item.label = new_label
                self.refresh_selected_panel()
                self.redraw()

    def refresh_selected_panel(self, live=False):
        if not self.selected_ids:
            self.sel_type_var.set("-")
            self.sel_label_var.set("")
            self.sel_text_var.set("")
            self.sel_x_var.set("0")
            self.sel_y_var.set("0")
            self.sel_w_var.set("0")
            self.sel_h_var.set("0")
            self.sel_font_var.set(str(DEFAULT_TEXT_FONT_SIZE))
            self.sel_font_family_var.set(self.default_font_family.get())
            self.sel_show_label_var.set(True)
            self.sel_align_var.set("left")
            return

        if len(self.selected_ids) > 1:
            self.sel_type_var.set(f"{len(self.selected_ids)} items")
            anchor = self.get_item_by_id(self.anchor_selected_id) if self.anchor_selected_id else None
            if anchor:
                self.sel_x_var.set(str(anchor.x))
                self.sel_y_var.set(str(anchor.y))
                self.sel_w_var.set(str(anchor.w))
                self.sel_h_var.set(str(anchor.h))
            return

        sel = self.get_item_by_id(self.selected_ids[0])
        if sel is None:
            return

        self.sel_type_var.set(sel.kind)
        self.sel_x_var.set(str(sel.x))
        self.sel_y_var.set(str(sel.y))
        self.sel_w_var.set(str(sel.w))
        self.sel_h_var.set(str(sel.h))

        if isinstance(sel, PanelItem):
            self.sel_label_var.set(sel.label)
            self.sel_text_var.set("")
            self.sel_font_var.set(str(sel.label_font_size))
            self.sel_font_family_var.set(sel.label_font_family)
            self.sel_show_label_var.set(sel.show_label)
            self.sel_align_var.set("left")
        else:
            self.sel_label_var.set("")
            self.sel_text_var.set(sel.text)
            self.sel_font_var.set(str(sel.font_size))
            self.sel_font_family_var.set(sel.font_family)
            self.sel_show_label_var.set(True)
            self.sel_align_var.set(sel.align)

        if not live:
            self.set_status(f"Selected {sel.kind} #{sel.id}")

    def apply_selected_properties(self):
        if not self.selected_ids:
            return
        self.save_undo_state()
        font_family = self.sel_font_family_var.get()
        align = self.sel_align_var.get()
        try:
            x = int(float(self.sel_x_var.get()))
            y = int(float(self.sel_y_var.get()))
            w = max(MIN_PANEL_SIZE, int(float(self.sel_w_var.get())))
            h = max(MIN_PANEL_SIZE, int(float(self.sel_h_var.get())))
            font_size = max(6, int(float(self.sel_font_var.get())))
        except ValueError:
            messagebox.showerror("Invalid input", "Position, size, and font size must be numeric.")
            return

        if len(self.selected_ids) == 1:
            sel = self.get_item_by_id(self.selected_ids[0])
            if sel is None:
                return
            sel.x, sel.y, sel.w, sel.h = x, y, w, h
            if isinstance(sel, PanelItem):
                sel.label = self.sel_label_var.get()
                sel.label_font_size = font_size
                sel.label_font_family = font_family
                sel.show_label = self.sel_show_label_var.get()
            else:
                sel.text = self.sel_text_var.get()
                sel.font_size = font_size
                sel.font_family = font_family
                sel.align = align
                if sel.text.strip().lower().startswith("caption"):
                    self.fit_text_box_height(sel, maybe_expand_canvas=True)
        else:
            anchor = self.get_item_by_id(self.anchor_selected_id) if self.anchor_selected_id else None
            if anchor:
                anchor.x, anchor.y, anchor.w, anchor.h = x, y, w, h

        self.redraw()
        self.set_status("Properties applied")

    def pick_selected_text_colour(self):
        if len(self.selected_ids) != 1:
            return
        item = self.get_item_by_id(self.selected_ids[0])
        if not isinstance(item, TextItem):
            return
        color = colorchooser.askcolor(color=item.fill, title="Choose text colour")[1]
        if color:
            self.save_undo_state()
            item.fill = color
            self.redraw()

    def bring_to_front(self):
        selected = self.get_selected_items()
        if not selected:
            return
        self.save_undo_state()
        max_z = max((i.z_index for i in self.items), default=0)
        for idx, sel in enumerate(selected):
            sel.z_index = max_z + idx + 1
        self.redraw()

    def send_to_back(self):
        selected = self.get_selected_items()
        if not selected:
            return
        self.save_undo_state()
        min_z = min((i.z_index for i in self.items), default=0)
        for idx, sel in enumerate(selected):
            sel.z_index = min_z - len(selected) + idx
        self.redraw()

    def duplicate_selected(self):
        selected = self.get_selected_items()
        if not selected:
            return
        self.save_undo_state()
        new_ids = []
        for sel in selected:
            if isinstance(sel, PanelItem):
                dup = PanelItem(
                    id=self.get_next_id(),
                    source_path=sel.source_path,
                    pil_image=sel.pil_image.copy() if sel.pil_image else None,
                    original_size=sel.original_size,
                    x=sel.x + 20, y=sel.y + 20, w=sel.w, h=sel.h,
                    label=sel.label, show_label=sel.show_label,
                    label_font_size=sel.label_font_size,
                    label_font_family=sel.label_font_family,
                    label_offset_x=sel.label_offset_x,
                    label_offset_y=sel.label_offset_y,
                    border_width=sel.border_width,
                    border_color=sel.border_color,
                    z_index=max((i.z_index for i in self.items), default=0) + 1,
                    group_id=sel.group_id
                )
            else:
                dup = TextItem(
                    id=self.get_next_id(),
                    text=sel.text,
                    x=sel.x + 20, y=sel.y + 20, w=sel.w, h=sel.h,
                    font_size=sel.font_size,
                    font_family=sel.font_family,
                    bold=sel.bold, fill=sel.fill, align=sel.align,
                    background=sel.background, outline=sel.outline,
                    z_index=max((i.z_index for i in self.items), default=0) + 1,
                    group_id=sel.group_id
                )
            self.items.append(dup)
            new_ids.append(dup.id)
        self.selected_ids = new_ids
        self.anchor_selected_id = new_ids[-1] if new_ids else None
        self.refresh_selected_panel()
        self.redraw()

    def delete_selected(self):
        if not self.selected_ids:
            return
        self.save_undo_state()
        selected_set = set(self.selected_ids)
        self.items = [i for i in self.items if i.id not in selected_set]
        self.selected_ids = []
        self.anchor_selected_id = None
        self.refresh_selected_panel()
        self.redraw()
        self.set_status("Deleted selected item(s)")

    def regenerate_labels(self, silent=False):
        panels = [i for i in self.items if isinstance(i, PanelItem)]
        if self.keep_label_order.get():
            panels = sorted(panels, key=lambda p: p.id)
        else:
            panels = sorted(panels, key=lambda p: (p.y, p.x))
        for idx, panel in enumerate(panels):
            panel.label = next_panel_label(idx)
            panel.label_font_size = self.default_label_size.get()
            panel.label_font_family = self.default_font_family.get()
        self.redraw()
        if not silent:
            self.set_status("Regenerated panel labels")

    def toggle_labels(self):
        panels = [i for i in self.items if isinstance(i, PanelItem)]
        if not panels:
            return
        self.save_undo_state()
        turn_on = any(not p.show_label for p in panels)
        for p in panels:
            p.show_label = turn_on
        self.redraw()

    def apply_default_label_size(self):
        size = self.default_label_size.get()
        for item in self.items:
            if isinstance(item, PanelItem):
                item.label_font_size = size
        self.redraw()

    def auto_grid_if_reasonable(self):
        panels = [i for i in self.items if isinstance(i, PanelItem)]
        if 2 <= len(panels) <= 20:
            self.auto_grid()

    def get_group_members(self, group_id: int) -> List[CanvasItem]:
        return [i for i in self.items if i.group_id == group_id]

    def get_selection_units(self) -> List[List[CanvasItem]]:
        selected = self.get_selected_items()
        if not selected:
            return []
        units = []
        used = set()
        for item in selected:
            if item.id in used:
                continue
            if item.group_id is not None:
                members = [m for m in self.get_group_members(item.group_id) if m.id in self.selected_ids]
                if len(members) > 1:
                    units.append(members)
                    used.update(m.id for m in members)
                else:
                    units.append([item])
                    used.add(item.id)
            else:
                units.append([item])
                used.add(item.id)
        return units

    def unit_bbox(self, unit: List[CanvasItem]):
        x1 = min(i.x for i in unit)
        y1 = min(i.y for i in unit)
        x2 = max(i.x + i.w for i in unit)
        y2 = max(i.y + i.h for i in unit)
        return x1, y1, x2, y2

    def move_unit_to(self, unit: List[CanvasItem], new_x: Optional[int] = None, new_y: Optional[int] = None):
        x1, y1, _, _ = self.unit_bbox(unit)
        dx = 0 if new_x is None else new_x - x1
        dy = 0 if new_y is None else new_y - y1
        for item in unit:
            item.x += dx
            item.y += dy

    def group_selected(self):
        if len(self.selected_ids) < 2:
            return
        self.save_undo_state()
        gid = self.get_next_group_id()
        for item in self.get_selected_items():
            item.group_id = gid
        self.redraw()
        self.set_status("Grouped selected items")

    def ungroup_selected(self):
        selected = self.get_selected_items()
        if not selected:
            return
        self.save_undo_state()
        for item in selected:
            item.group_id = None
        self.redraw()
        self.set_status("Ungrouped selected items")

    def apply_custom_grid(self):
        panels = [i for i in self.items if isinstance(i, PanelItem)]
        if not panels:
            return
        try:
            rows = max(1, int(self.grid_rows_entry.get()))
            cols = max(1, int(self.grid_cols_entry.get()))
        except ValueError:
            messagebox.showerror("Invalid grid", "Rows and columns must be integers.")
            return
        self.save_undo_state()
        self.grid_layout(panels, rows, cols)
        self.redraw()
        self.set_status(f"Applied {rows} × {cols} grid")

    def auto_grid(self):
        panels = [i for i in self.items if isinstance(i, PanelItem)]
        if not panels:
            return
        self.save_undo_state()
        n = len(panels)
        cols = math.ceil(math.sqrt(n))
        rows = math.ceil(n / cols)
        self.grid_layout(panels, rows, cols)
        self.redraw()
        self.set_status(f"Auto-arranged {n} panel(s)")

    def grid_layout(self, panels: List[PanelItem], rows: int, cols: int):
        margin = self.outer_margin.get()
        gap_x = self.gap_x.get()
        gap_y = self.gap_y.get()

        avail_w = self.canvas_w - 2 * margin - (cols - 1) * gap_x
        avail_h = self.canvas_h - 2 * margin - (rows - 1) * gap_y

        cell_w = max(50, avail_w // cols)
        cell_h = max(50, avail_h // rows)

        panels_sorted = sorted(panels, key=lambda p: p.id if self.keep_label_order.get() else (p.y, p.x))

        for idx, p in enumerate(panels_sorted):
            r = idx // cols
            c = idx % cols
            x = margin + c * (cell_w + gap_x)
            y = margin + r * (cell_h + gap_y)
            ow, oh = p.original_size
            new_w, new_h = fit_size_keep_aspect(ow, oh, cell_w, cell_h)
            p.x = x + (cell_w - new_w) // 2
            p.y = y + (cell_h - new_h) // 2
            p.w = new_w
            p.h = new_h

    def distribute_h(self):
        units = self.get_selection_units()
        if len(units) < 3:
            units = [[p] for p in sorted([i for i in self.items if isinstance(i, PanelItem)], key=lambda z: z.x)]
        if len(units) < 3:
            return
        self.save_undo_state()
        units = sorted(units, key=lambda u: self.unit_bbox(u)[0])
        left = min(self.unit_bbox(u)[0] for u in units)
        right = max(self.unit_bbox(u)[2] for u in units)
        total_w = sum(self.unit_bbox(u)[2] - self.unit_bbox(u)[0] for u in units)
        gap = max(SAFE_GAP, int(round((right - left - total_w) / (len(units) - 1))))
        cur_x = left
        for u in units:
            self.move_unit_to(u, new_x=cur_x)
            ux1, _, ux2, _ = self.unit_bbox(u)
            cur_x = ux2 + gap
        self.resolve_all_overlaps(redraw=False)
        self.redraw()

    def distribute_v(self):
        units = self.get_selection_units()
        if len(units) < 3:
            units = [[p] for p in sorted([i for i in self.items if isinstance(i, PanelItem)], key=lambda z: z.y)]
        if len(units) < 3:
            return
        self.save_undo_state()
        units = sorted(units, key=lambda u: self.unit_bbox(u)[1])
        top = min(self.unit_bbox(u)[1] for u in units)
        bottom = max(self.unit_bbox(u)[3] for u in units)
        total_h = sum(self.unit_bbox(u)[3] - self.unit_bbox(u)[1] for u in units)
        gap = max(SAFE_GAP, int(round((bottom - top - total_h) / (len(units) - 1))))
        cur_y = top
        for u in units:
            self.move_unit_to(u, new_y=cur_y)
            _, uy1, _, uy2 = self.unit_bbox(u)
            cur_y = uy2 + gap
        self.resolve_all_overlaps(redraw=False)
        self.redraw()

    def align_to_anchor(self, where: str):
        units = self.get_selection_units()
        if len(units) < 2:
            return
        anchor = self.get_item_by_id(self.anchor_selected_id) if self.anchor_selected_id else None
        if anchor is None:
            return
        self.save_undo_state()
        anchor_unit = None
        for u in units:
            if any(i.id == anchor.id for i in u):
                anchor_unit = u
                break
        if anchor_unit is None:
            return
        ax1, ay1, ax2, ay2 = self.unit_bbox(anchor_unit)
        for u in units:
            if u == anchor_unit:
                continue
            ux1, uy1, ux2, uy2 = self.unit_bbox(u)
            if where == "left":
                self.move_unit_to(u, new_x=ax1)
            elif where == "right":
                self.move_unit_to(u, new_x=ax2 - (ux2 - ux1))
            elif where == "top":
                self.move_unit_to(u, new_y=ay1)
            elif where == "bottom":
                self.move_unit_to(u, new_y=ay2 - (uy2 - uy1))
        self.resolve_all_overlaps(redraw=False)
        self.redraw()

    def same_widths(self):
        selected = self.get_selected_items()
        if len(selected) < 2:
            return
        anchor = self.get_item_by_id(self.anchor_selected_id) if self.anchor_selected_id else selected[0]
        if anchor is None:
            return
        self.save_undo_state()
        w = anchor.w
        for item in selected:
            if item.id == anchor.id:
                continue
            if isinstance(item, PanelItem):
                ow, oh = item.original_size
                new_h = int(round(w * oh / ow)) if ow else item.h
                item.w = max(MIN_PANEL_SIZE, w)
                item.h = max(MIN_PANEL_SIZE, new_h)
            else:
                item.w = max(MIN_PANEL_SIZE, w)
        self.resolve_all_overlaps(redraw=False)
        self.redraw()

    def same_heights(self):
        selected = self.get_selected_items()
        if len(selected) < 2:
            return
        anchor = self.get_item_by_id(self.anchor_selected_id) if self.anchor_selected_id else selected[0]
        if anchor is None:
            return
        self.save_undo_state()
        h = anchor.h
        for item in selected:
            if item.id == anchor.id:
                continue
            if isinstance(item, PanelItem):
                ow, oh = item.original_size
                new_w = int(round(h * ow / oh)) if oh else item.w
                item.w = max(MIN_PANEL_SIZE, new_w)
                item.h = max(MIN_PANEL_SIZE, h)
            else:
                item.h = max(MIN_PANEL_SIZE, h)
        self.resolve_all_overlaps(redraw=False)
        self.redraw()

    def content_bbox(self) -> Optional[Tuple[int, int, int, int]]:
        if not self.items:
            return None
        x1 = min(i.x for i in self.items)
        y1 = min(i.y for i in self.items)
        x2 = max(i.x + i.w for i in self.items)
        y2 = max(i.y + i.h for i in self.items)
        return x1, y1, x2, y2

    def crop_canvas_to_content(self):
        bbox = self.content_bbox()
        if bbox is None:
            return
        self.save_undo_state()
        margin = self.outer_margin.get()
        x1, y1, x2, y2 = bbox
        shift_x = margin - x1
        shift_y = margin - y1
        for item in self.items:
            item.x += shift_x
            item.y += shift_y
        self.canvas_w = max(100, x2 - x1 + 2 * margin)
        self.canvas_h = max(100, y2 - y1 + 2 * margin)
        self.canvas_w_entry.delete(0, "end")
        self.canvas_w_entry.insert(0, str(self.canvas_w))
        self.canvas_h_entry.delete(0, "end")
        self.canvas_h_entry.insert(0, str(self.canvas_h))
        self.canvas.config(scrollregion=(0, 0, self.canvas_w, self.canvas_h))
        self.redraw()
        self.set_status("Canvas cropped to content")

    def resolve_all_overlaps(self, redraw=True):
        panels = sorted([i for i in self.items if isinstance(i, PanelItem)], key=lambda p: (p.y, p.x))
        changed = True
        rounds = 0
        while changed and rounds < 30:
            changed = False
            rounds += 1
            for i in range(len(panels)):
                for j in range(i + 1, len(panels)):
                    a = panels[i]
                    b = panels[j]
                    if rects_overlap(a.bbox(), b.bbox(), pad=0):
                        new_x = a.x
                        if b.x >= a.x:
                            b.x = a.x + a.w + max(self.gap_x.get(), SAFE_GAP)
                        else:
                            b.x = max(0, a.x - b.w - max(self.gap_x.get(), SAFE_GAP))
                        if b.x < self.outer_margin.get() or b.x + b.w > self.canvas_w - self.outer_margin.get():
                            b.x = new_x
                            b.y = a.y + a.h + max(self.gap_y.get(), SAFE_GAP)
                        changed = True
        if redraw:
            self.redraw()

    def toggle_erase_mode(self):
        self.erase_mode = not self.erase_mode
        self.erase_start = None
        self.erase_current = None
        self.set_status("Erase rectangle mode ON" if self.erase_mode else "Erase rectangle mode OFF")
        self.redraw()

    def apply_erase_rectangle(self, x1, y1, x2, y2):
        if abs(x2 - x1) < 2 or abs(y2 - y1) < 2:
            return
        rect = (min(x1, x2), min(y1, y2), max(x1, x2), max(y1, y2))
        target = None
        for item in sorted(self.items, key=lambda z: z.z_index, reverse=True):
            if isinstance(item, PanelItem):
                ix1, iy1, ix2, iy2 = item.bbox()
                if rects_overlap(rect, (ix1, iy1, ix2, iy2)):
                    target = item
                    break
        if target is None or target.pil_image is None:
            return

        ix1, iy1, ix2, iy2 = target.bbox()
        rx1 = clamp(rect[0], ix1, ix2)
        ry1 = clamp(rect[1], iy1, iy2)
        rx2 = clamp(rect[2], ix1, ix2)
        ry2 = clamp(rect[3], iy1, iy2)
        if rx2 <= rx1 or ry2 <= ry1:
            return

        src_w, src_h = target.pil_image.size
        sx1 = int(round((rx1 - target.x) * src_w / target.w))
        sy1 = int(round((ry1 - target.y) * src_h / target.h))
        sx2 = int(round((rx2 - target.x) * src_w / target.w))
        sy2 = int(round((ry2 - target.y) * src_h / target.h))

        draw = ImageDraw.Draw(target.pil_image)
        draw.rectangle([sx1, sy1, sx2, sy2], fill="white")
        self.set_status("Erased selected rectangle from panel")

    def fit_text_box_height(self, item: TextItem, maybe_expand_canvas=False):
        img = Image.new("RGB", (max(100, item.w), 2000), "white")
        draw = ImageDraw.Draw(img)
        font = get_font(item.font_family, item.font_size, item.bold)
        lines = self.wrap_text_to_width(draw, item.text, font, max(20, item.w - 12))
        y = 6
        for line in lines:
            bbox = draw.textbbox((0, 0), line, font=font)
            y += (bbox[3] - bbox[1]) + 4
        item.h = max(item.h, y + 8)
        if maybe_expand_canvas:
            needed_h = item.y + item.h + self.outer_margin.get()
            if needed_h > self.canvas_h:
                self.canvas_h = needed_h
                self.canvas_h_entry.delete(0, "end")
                self.canvas_h_entry.insert(0, str(self.canvas_h))
                self.canvas.config(scrollregion=(0, 0, self.canvas_w, self.canvas_h))

    def render_final_image(self, dpi: int) -> Image.Image:
        scale = max(1.0, dpi / 100.0)
        out_w = max(1, int(round(self.canvas_w * scale)))
        out_h = max(1, int(round(self.canvas_h * scale)))
        out = Image.new("RGB", (out_w, out_h), "white")
        draw = ImageDraw.Draw(out)

        for item in sorted(self.items, key=lambda x: x.z_index):
            if isinstance(item, PanelItem):
                if item.pil_image is None:
                    continue
                ix = int(round(item.x * scale))
                iy = int(round(item.y * scale))
                iw = max(1, int(round(item.w * scale)))
                ih = max(1, int(round(item.h * scale)))
                resized = item.pil_image.resize((iw, ih), Image.LANCZOS).convert("RGB")
                out.paste(resized, (ix, iy))
                if item.border_width > 0:
                    draw.rectangle([ix, iy, ix + iw, iy + ih],
                                   outline=item.border_color,
                                   width=max(1, int(round(item.border_width * scale))))
                if item.show_label and item.label:
                    font = get_font(item.label_font_family, max(6, int(round(item.label_font_size * scale))), bold=True)
                    draw.text((int(round((item.x + item.label_offset_x) * scale)),
                               int(round((item.y + item.label_offset_y) * scale))),
                              item.label, fill="black", font=font)
            else:
                ix = int(round(item.x * scale))
                iy = int(round(item.y * scale))
                iw = max(1, int(round(item.w * scale)))
                ih = max(1, int(round(item.h * scale)))

                if item.background:
                    draw.rectangle([ix, iy, ix + iw, iy + ih],
                                   fill=item.background,
                                   outline=item.outline or None)
                elif item.outline:
                    draw.rectangle([ix, iy, ix + iw, iy + ih],
                                   outline=item.outline,
                                   width=max(1, int(round(scale))))

                font = get_font(item.font_family, max(6, int(round(item.font_size * scale))), item.bold)
                scaled_item = TextItem(
                    id=item.id, text=item.text, x=ix, y=iy, w=iw, h=ih,
                    font_size=max(6, int(round(item.font_size * scale))),
                    font_family=item.font_family,
                    bold=item.bold, fill=item.fill, align=item.align,
                    background=item.background, outline=item.outline, z_index=item.z_index
                )
                self._draw_wrapped_text_pil(draw, scaled_item, font)

        if self.auto_trim_on_export.get():
            out = self.trim_image(out, pad=max(0, int(round(self.outer_margin.get() * scale))))
        return out

    def _draw_wrapped_text_pil(self, draw: ImageDraw.ImageDraw, item: TextItem, font):
        text = item.text
        max_width = max(20, item.w - 12)
        lines = self.wrap_text_to_width(draw, text, font, max_width)
        y = item.y + 6
        for line in lines:
            bbox = draw.textbbox((0, 0), line, font=font)
            tw = bbox[2] - bbox[0]
            th = bbox[3] - bbox[1]
            if item.align == "center":
                x = item.x + (item.w - tw) / 2
            elif item.align == "right":
                x = item.x + item.w - 6 - tw
            else:
                x = item.x + 6
            draw.text((x, y), line, fill=item.fill, font=font)
            y += th + 4

    @staticmethod
    def wrap_text_to_width(draw: ImageDraw.ImageDraw, text: str, font, max_width: int) -> List[str]:
        if "\n" in text:
            lines = []
            for chunk in text.splitlines():
                lines.extend(FigureBoardApp.wrap_text_to_width(draw, chunk, font, max_width))
            return lines or [""]
        words = text.split()
        if not words:
            return [""]
        lines = []
        current = words[0]
        for word in words[1:]:
            test = current + " " + word
            bbox = draw.textbbox((0, 0), test, font=font)
            if bbox[2] - bbox[0] <= max_width:
                current = test
            else:
                lines.append(current)
                current = word
        lines.append(current)
        return lines

    @staticmethod
    def trim_image(img: Image.Image, pad: int = 0) -> Image.Image:
        bg = Image.new(img.mode, img.size, img.getpixel((0, 0)))
        diff = ImageChops.difference(img, bg)
        bbox = diff.getbbox()
        if bbox is None:
            return img
        x1, y1, x2, y2 = bbox
        x1 = max(0, x1 - pad)
        y1 = max(0, y1 - pad)
        x2 = min(img.width, x2 + pad)
        y2 = min(img.height, y2 + pad)
        return img.crop((x1, y1, x2, y2))

    def export_image(self, fmt: str):
        if not self.items:
            messagebox.showinfo("Nothing to export", "Add some panels or text first.")
            return

        ext_map = {"PNG": ".png", "TIFF": ".tiff"}
        path = filedialog.asksaveasfilename(
            title=f"Export {fmt}",
            defaultextension=ext_map[fmt],
            filetypes=[(f"{fmt} file", f"*{ext_map[fmt]}")]
        )
        if not path:
            return

        try:
            dpi = max(72, int(self.export_dpi.get()))
            img = self.render_final_image(dpi=dpi)
            save_kwargs = {"dpi": (dpi, dpi)}
            if fmt == "TIFF":
                save_kwargs["compression"] = "tiff_deflate"
            img.save(path, format=fmt, **save_kwargs)
            self.set_status(f"Exported {fmt}: {path}")
        except Exception as e:
            messagebox.showerror("Export error", f"Could not export file.\n\n{e}")

    def export_pdf(self):
        if not self.items:
            messagebox.showinfo("Nothing to export", "Add some panels or text first.")
            return

        path = filedialog.asksaveasfilename(
            title="Export PDF",
            defaultextension=".pdf",
            filetypes=[("PDF file", "*.pdf")]
        )
        if not path:
            return

        try:
            dpi = max(72, int(self.export_dpi.get()))
            img = self.render_final_image(dpi=dpi)
            bio = io.BytesIO()
            img.save(bio, format="PNG", dpi=(dpi, dpi))
            png_bytes = bio.getvalue()

            width_pt = img.width / dpi * 72.0
            height_pt = img.height / dpi * 72.0

            doc = fitz.open()
            page = doc.new_page(width=width_pt, height=height_pt)
            page.insert_image(fitz.Rect(0, 0, width_pt, height_pt), stream=png_bytes)
            doc.save(path, deflate=True, garbage=4)
            doc.close()
            self.set_status(f"Exported PDF: {path}")
        except Exception as e:
            messagebox.showerror("Export error", f"Could not export PDF.\n\n{e}")


def main():
    if DND_AVAILABLE:
        root = TkinterDnD.Tk()
    else:
        root = tk.Tk()

    try:
        style = ttk.Style(root)
        if "vista" in style.theme_names():
            style.theme_use("vista")
    except Exception:
        pass

    app = FigureBoardApp(root)
    if not DND_AVAILABLE:
        messagebox.showwarning(
            "Drag-and-drop unavailable",
            "tkinterdnd2 is not available.\n\nYou can still use the app with the 'Add image/PDF files' button.\n\nInstall with:\npip install tkinterdnd2"
        )
    root.mainloop()


if __name__ == "__main__":
    main()